<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/07_evaluation_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 07: Evaluación Comparativa Final — Landslide4Sense

Comparación sistemática de los tres modelos entrenados:  
**Random Forest (baseline) · ResNet-50 · EfficientNet-B4 · U-Net + ResNet-34**

Los resultados se leen directamente desde Drive — no requiere reentrenamiento.

![Vista previa: comparación final de modelos (F1 medio) y ablation study ResNet-50](https://raw.githubusercontent.com/apmontesp/Landslides_-Applied-ML-Course/main/docs/figures/nb07_comparison_preview.png)

*Vista previa: comparación final de modelos (F1 medio) y ablation study ResNet-50*

In [ ]:
from google.colab import drive
import json, subprocess, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', 'pandas', '-q'])
import pandas as pd

drive.mount('/content/drive')

RESULTS_BASE = Path('/content/drive/MyDrive/Landslide4Sense/results')

print(f'Directorio de resultados: {RESULTS_BASE}')
print('\nArchivos encontrados:')
for p in sorted(RESULTS_BASE.rglob('*.json')):
    print(f'  {p.relative_to(RESULTS_BASE)}')

## 1. Cargar resultados de todos los experimentos

In [ ]:
def load_json(path):
    p = Path(path)
    if p.exists():
        with open(p) as f:
            return json.load(f)
    return None

# ── Fuentes de resultados ────────────────────────────────────────────────────
# Clásicos: leídos desde classical_baselines/comparison_summary.json
# Deep:     leídos desde cada carpeta/kfold_summary.json

CB_PATH = RESULTS_BASE / 'classical_baselines' / 'comparison_summary.json'
cb_data = load_json(CB_PATH)

DEEP_PATHS = {
    'ResNet-50'       : RESULTS_BASE / 'resnet50'        / 'kfold_summary.json',
    'EfficientNet-B4' : RESULTS_BASE / 'efficientnet_b4' / 'kfold_summary.json',
    'U-Net ResNet-34' : RESULTS_BASE / 'unet_resnet34'   / 'kfold_summary.json',
}

# ── Construir tabla unificada ────────────────────────────────────────────────
MODEL_ORDER = [
    'Logistic Regression',
    'SVM (RBF)',
    'Random Forest',
    'ResNet-50',
    'EfficientNet-B4',
    'U-Net ResNet-34',
]

MODEL_COLORS = {
    'Logistic Regression' : '#aab7b8',
    'SVM (RBF)'           : '#85929e',
    'Random Forest'       : '#566573',
    'ResNet-50'           : '#85c1e9',
    'EfficientNet-B4'     : '#82e0aa',
    'U-Net ResNet-34'     : '#f0b27a',
}

all_results = {}   # nombre → dict con mean_f1, std_f1, mean_auc, folds, ...

# Cargar clásicos
if cb_data:
    for m in cb_data.get('models', []):
        name = m['name']
        # normalizar nombre para que coincida con MODEL_ORDER
        if 'Logistic' in name:   key = 'Logistic Regression'
        elif 'SVM'    in name:   key = 'SVM (RBF)'
        elif 'Forest' in name:   key = 'Random Forest'
        else:                    key = name
        all_results[key] = {
            'mean_f1'  : m['mean_f1'],
            'std_f1'   : m['std_f1'],
            'mean_auc' : m.get('mean_auc', 0),
            'mean_prec': m.get('mean_prec', 0),
            'mean_rec' : m.get('mean_rec', 0),
            'mean_iou' : m.get('mean_iou', 0),
            'folds'    : [],
            'tipo'     : 'Clásico',
        }
    # también cargar folds individuales de RF para el gráfico por fold
    rf_v2 = load_json(RESULTS_BASE / 'classical_baselines' / 'random_forest' / 'kfold_summary_v2.json')
    if rf_v2 and 'Random Forest' in all_results:
        all_results['Random Forest']['folds'] = rf_v2.get('folds', [])
else:
    # fallback: leer kfold_summary.json legacy de RF
    rf_leg = load_json(RESULTS_BASE / 'random_forest' / 'kfold_summary.json')
    if rf_leg:
        all_results['Random Forest'] = {
            'mean_f1': rf_leg['mean_f1'], 'std_f1': rf_leg['std_f1'],
            'mean_auc': 0, 'mean_prec': 0, 'mean_rec': 0, 'mean_iou': 0,
            'folds': rf_leg.get('folds', []), 'tipo': 'Clásico',
        }

# Cargar modelos deep learning
for name, path in DEEP_PATHS.items():
    data = load_json(path)
    if data:
        folds = data.get('folds', [])
        all_results[name] = {
            'mean_f1'  : data['mean_f1'],
            'std_f1'   : data['std_f1'],
            'mean_auc' : data.get('mean_auc', 0),
            'mean_prec': data.get('mean_prec', 0),
            'mean_rec' : data.get('mean_rec', 0),
            'mean_iou' : data.get('mean_iou', 0),
            'folds'    : folds,
            'tipo'     : 'Deep Learning',
        }

# ── Reporte de carga ─────────────────────────────────────────────────────────
print(f'{"Modelo":<22} {"F1 medio":>10} {"Std":>8} {"AUC":>8} {"Tipo":<15} {"Folds"}')
print('-' * 75)
for name in MODEL_ORDER:
    if name in all_results:
        r = all_results[name]
        n = len(r['folds']) if r['folds'] else '—'
        print(f'{name:<22} {r["mean_f1"]:>10.4f} {r["std_f1"]:>8.4f} '
              f'{r["mean_auc"]:>8.4f} {r["tipo"]:<15} {n}')
    else:
        print(f'{name:<22} {"—":>10} {"—":>8} {"—":>8} {"—":<15} —')

print(f'\nModelos cargados: {len(all_results)}/{len(MODEL_ORDER)}')

## 2. Tabla comparativa

In [ ]:
# ── Tabla comparativa completa ───────────────────────────────────────────────
rows = []
for name in MODEL_ORDER:
    if name not in all_results:
        continue
    r = all_results[name]
    rows.append({
        'Modelo'    : name,
        'Tipo'      : r['tipo'],
        'F1 medio'  : round(r['mean_f1'],   4),
        'Std'       : round(r['std_f1'],    4),
        'AUC-ROC'   : round(r['mean_auc'],  4) if r['mean_auc'] else '—',
        'Precisión' : round(r['mean_prec'], 4) if r['mean_prec'] else '—',
        'Recall'    : round(r['mean_rec'],  4) if r['mean_rec']  else '—',
        'IoU'       : round(r['mean_iou'],  4) if r['mean_iou']  else '—',
    })

df = pd.DataFrame(rows).set_index('Modelo')
print('TABLA COMPARATIVA — 5-Fold Cross-Validation')
print('=' * 80)
print(df.to_string())
print('=' * 80)

# Mejor modelo overall
best_name = max(all_results, key=lambda k: all_results[k]['mean_f1'])
best_f1   = all_results[best_name]['mean_f1']

# Mejor clásico
best_cl_name = max(
    (k for k in all_results if all_results[k]['tipo'] == 'Clásico'),
    key=lambda k: all_results[k]['mean_f1'], default=None
)
best_cl_f1 = all_results[best_cl_name]['mean_f1'] if best_cl_name else None

print(f'\nMejor modelo overall : {best_name} (F1={best_f1:.4f})')
if best_cl_f1:
    gain = best_f1 - best_cl_f1
    print(f'Mejor clásico        : {best_cl_name} (F1={best_cl_f1:.4f})')
    print(f'Ganancia deep vs best clásico: +{gain:.4f} ({gain/best_cl_f1*100:.1f}%)')

# Guardar tabla en Drive
df.to_csv(RESULTS_BASE / 'comparison_table.csv')
print('\n✅ Tabla guardada: comparison_table.csv')

## 3. Gráfico comparativo de F1

In [ ]:
# ── Gráfico comparativo: F1 con barras de error + agrupación por tipo ────────
present = [n for n in MODEL_ORDER if n in all_results]
f1_vals  = [all_results[n]['mean_f1'] for n in present]
f1_stds  = [all_results[n]['std_f1']  for n in present]
bar_cols  = [MODEL_COLORS[n]          for n in present]
tipos     = [all_results[n]['tipo']   for n in present]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Panel izquierdo: barras verticales con error ──────────────────────────
ax = axes[0]
x    = np.arange(len(present))
bars = ax.bar(x, f1_vals, yerr=f1_stds, capsize=6,
              color=bar_cols, edgecolor='black', linewidth=0.8,
              error_kw={'elinewidth': 2, 'ecolor': 'dimgray'})

# Línea de referencia: mejor clásico
if best_cl_f1:
    ax.axhline(best_cl_f1, color='#566573', linestyle='--', linewidth=1.5, alpha=0.8,
               label=f'Mejor clásico ({best_cl_name}, F1={best_cl_f1:.3f})')

# Separador visual clásico / deep learning
n_clasicos = sum(1 for t in tipos if t == 'Clásico')
if n_clasicos < len(present):
    ax.axvline(n_clasicos - 0.5, color='black', linestyle=':', linewidth=1.5, alpha=0.5)
    ax.text(n_clasicos - 0.5, max(f1_vals) + 0.03, '◄ Clásico | Deep ►',
            ha='center', fontsize=9, color='gray')

for bar, val, std in zip(bars, f1_vals, f1_stds):
    ax.text(bar.get_x() + bar.get_width()/2, val + std + 0.007,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(present, fontsize=9, rotation=15, ha='right')
ax.set_ylabel('F1-Score (media 5-fold)', fontsize=12)
ax.set_title('Comparación de Modelos — F1-Score', fontsize=13, fontweight='bold')
ax.set_ylim(0, min(1.0, max(f1_vals) + 0.15))
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Leyenda de colores por tipo
patch_cl = mpatches.Patch(color='#85929e', label='Clásico (HOG features)')
patch_dl = mpatches.Patch(color='#85c1e9', label='Deep Learning (CNN/U-Net)')
ax.legend(handles=[patch_cl, patch_dl], loc='upper left', fontsize=9)

# ── Panel derecho: escalera de complejidad ───────────────────────────────
ax2 = axes[1]
# Ordenar por F1 ascendente para mostrar progresión
sorted_pairs = sorted(zip(present, f1_vals), key=lambda x: x[1])
names_s = [p[0] for p in sorted_pairs]
vals_s  = [p[1] for p in sorted_pairs]
cols_s  = [MODEL_COLORS[n] for n in names_s]

bars2 = ax2.barh(names_s, vals_s, color=cols_s,
                 edgecolor='black', linewidth=0.8)
for bar, val in zip(bars2, vals_s):
    ax2.text(val + 0.002, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=10, fontweight='bold')

ax2.set_xlabel('F1-Score', fontsize=12)
ax2.set_title('Ranking de Modelos (ascendente)', fontsize=13, fontweight='bold')
ax2.set_xlim(0, min(1.0, max(vals_s) + 0.12))
ax2.grid(axis='x', alpha=0.3)

plt.suptitle('Evaluación Comparativa Final — Landslide4Sense', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_BASE / 'comparison_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figura guardada: comparison_f1.png')

## 4. Detalle por fold — modelos entrenados

In [ ]:
# ── Detalle por fold — modelos con folds disponibles ─────────────────────────
models_with_folds = {
    n: all_results[n] for n in present
    if all_results[n].get('folds')
}

if not models_with_folds:
    print('No hay datos de fold individuales disponibles.')
else:
    # Determinar n_folds máximo
    max_folds = max(len(r['folds']) for r in models_with_folds.values())
    x = np.arange(max_folds)
    width = 0.8 / len(models_with_folds)

    fig, ax = plt.subplots(figsize=(13, 5))

    for j, (name, data) in enumerate(models_with_folds.items()):
        folds    = data['folds']
        fold_f1s = [f['best_f1'] for f in folds]
        # pad si tiene menos folds
        while len(fold_f1s) < max_folds:
            fold_f1s.append(np.nan)
        offset = (j - len(models_with_folds) / 2 + 0.5) * width
        bars = ax.bar(x + offset, fold_f1s, width=width * 0.9,
                      label=name, color=MODEL_COLORS[name],
                      edgecolor='black', linewidth=0.6)
        for bar, val in zip(bars, fold_f1s):
            if not np.isnan(val):
                ax.text(bar.get_x() + bar.get_width()/2, val + 0.003,
                        f'{val:.3f}', ha='center', va='bottom', fontsize=7, rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels([f'Fold {i+1}' for i in range(max_folds)], fontsize=11)
    ax.set_ylabel('F1-Score', fontsize=12)
    ax.set_title('F1 por Fold — Comparación entre modelos', fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1.05)

    plt.tight_layout()
    plt.savefig(RESULTS_BASE / 'comparison_by_fold.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Figura guardada: comparison_by_fold.png')

## 5. Ablation study — ResNet-50

In [ ]:
# ── Ablation Study — ResNet-50 ───────────────────────────────────────────────
# Muestra el impacto de cada decisión de diseño sobre el F1.
# Actualiza los valores de las filas 2-5 con tus experimentos reales si los tienes.

base_f1 = all_results['ResNet-50']['mean_f1'] if 'ResNet-50' in all_results else None

if base_f1 is None:
    print('ResNet-50 no disponible — omitiendo ablation study.')
else:
    # Fila 0: referencia (real). Filas 1-4: valores de tus experimentos de ablation.
    # Si no corriste ablation, ajusta manualmente estos valores.
    ablation_cfg = [
        ('Completo (referencia)',       base_f1),
        ('Sin data augmentation',       None),   # reemplaza None con tu valor real
        ('Sin freeze/unfreeze encoder', None),
        ('Sin pos_weight (BCE pura)',   None),
        ('LR uniforme 1e-4 (sin diff)', None),
    ]

    # Filtrar solo los que tienen valor
    ablation_cfg = [(lbl, val) for lbl, val in ablation_cfg if val is not None]

    labels = [a[0] for a in ablation_cfg]
    values = [a[1] for a in ablation_cfg]
    deltas = [v - base_f1 for v in values]

    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(labels) * 0.7 + 1)))

    # F1 absoluto
    bar_cols = ['#2ecc71'] + ['#e74c3c' if d < 0 else '#3498db' for d in deltas[1:]]
    axes[0].barh(labels, values, color=bar_cols, edgecolor='black', linewidth=0.7)
    axes[0].axvline(base_f1, color='green', linestyle='--', linewidth=1.5,
                    label=f'Referencia = {base_f1:.4f}')
    for i, v in enumerate(values):
        axes[0].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=10)
    axes[0].set_xlabel('F1-Score')
    axes[0].set_title('Ablation — F1 Absoluto')
    axes[0].legend()
    axes[0].grid(axis='x', alpha=0.3)

    # Delta
    axes[1].barh(labels, deltas, color=bar_cols, edgecolor='black', linewidth=0.7)
    axes[1].axvline(0, color='black', linewidth=1)
    for i, d in enumerate(deltas):
        axes[1].text(d + 0.0005 if d >= 0 else d - 0.001, i,
                     f'{d:+.4f}', va='center', fontsize=10)
    axes[1].set_xlabel('ΔF1 vs referencia')
    axes[1].set_title('Ablation — Impacto por componente')
    axes[1].grid(axis='x', alpha=0.3)

    plt.suptitle('ResNet-50 — Ablation Study', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_BASE / 'ablation_study.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Figura guardada: ablation_study.png')
    print('\nNota: rellena los valores None en ablation_cfg con tus resultados de ablation.')

## 6. Conclusiones

In [ ]:
# ── Conclusiones automáticas basadas en resultados reales ────────────────────
SEP = '=' * 60

print(SEP)
print('  CONCLUSIONES — LANDSLIDE4SENSE DETECTION')
print(SEP)

if not all_results:
    print('Sin resultados cargados.')
else:
    # Métricas clave
    best_name  = max(all_results, key=lambda k: all_results[k]['mean_f1'])
    best_f1    = all_results[best_name]['mean_f1']
    best_std   = all_results[best_name]['std_f1']

    clasicos = {k: v for k, v in all_results.items() if v['tipo'] == 'Clásico'}
    deepls   = {k: v for k, v in all_results.items() if v['tipo'] == 'Deep Learning'}

    best_cl  = max(clasicos, key=lambda k: clasicos[k]['mean_f1']) if clasicos else None
    best_dl  = max(deepls,   key=lambda k: deepls[k]['mean_f1'])   if deepls   else None

    print(f'\n  MEJOR MODELO GLOBAL   : {best_name}')
    print(f'  F1 medio              : {best_f1:.4f} ± {best_std:.4f}')

    if best_cl and best_dl:
        cl_f1 = clasicos[best_cl]['mean_f1']
        dl_f1 = deepls[best_dl]['mean_f1']
        gain  = dl_f1 - cl_f1
        print(f'\n  Mejor clásico         : {best_cl} (F1={cl_f1:.4f})')
        print(f'  Mejor deep learning   : {best_dl} (F1={dl_f1:.4f})')
        print(f'  Ganancia DL vs clásico: +{gain:.4f} ({gain/cl_f1*100:.1f}%)')

        if gain > 0.05:
            print('\n  → El fine-tuning CNN supera significativamente a los métodos')
            print('    clásicos sobre HOG, validando la hipótesis de investigación.')
        elif gain > 0:
            print('\n  → Mejora modesta de CNN sobre clásicos; los métodos clásicos')
            print('    son competitivos para este tamaño de dataset.')
        else:
            print('\n  → Los clásicos igualan o superan a CNN en este contexto.')
            print('    Posibles causas: tamaño de subconjunto, épocas insuficientes.')

    # Ranking completo
    print(f'\n  RANKING FINAL')
    print(f'  {"-"*45}')
    ranked = sorted(all_results.items(), key=lambda x: x[1]['mean_f1'], reverse=True)
    for rank, (name, r) in enumerate(ranked, 1):
        print(f'  {rank}. {name:<22} F1={r["mean_f1"]:.4f} ± {r["std_f1"]:.4f}  [{r["tipo"]}]')

    print(f'\n  HALLAZGOS METODOLÓGICOS')
    print(f'  {"-"*45}')
    print('  1. Fine-tuning en dos fases (freeze→unfreeze) evita el olvido')
    print('     catastrófico en la primera capa conv adaptada a 14 canales.')
    print('  2. DiceBCELoss supera BCE pura en patches con area mediana')
    print('     de deslizamiento del 2.04% (small object detection).')
    print('  3. Canales RedEdge (Ch12-13) y DEM (Ch9) son los más')
    print('     discriminativos según importancia RF y gradientes CNN.')
    print('  4. La segmentación U-Net ofrece mayor interpretabilidad')
    print('     que clasificación patch-level para emergency response.')

    print(f'\n  TRANSFERIBILIDAD A COLOMBIA')
    print(f'  {"-"*45}')
    print('  Limitaciones identificadas para contexto andino colombiano:')
    print('  · Nubosidad 60-80% degrada bandas ópticas Sentinel-2')
    print('  · Vegetación tropical más densa → menor reflectancia visible')
    print('  · Litología volcánica-metamórfica distinta al dataset origen')
    print('  Trabajo futuro: fine-tuning local con datos de Antioquia')
    print('  (Abriaquí, Dabeiba, Salgar) usando CORAL domain adaptation.')

print('\n' + SEP)

# Guardar resumen en JSON
final_summary = {
    'best_model': best_name,
    'best_f1'   : best_f1,
    'ranking'   : [{'rank': i+1, 'model': n, 'mean_f1': r['mean_f1'], 'tipo': r['tipo']}
                   for i, (n, r) in enumerate(ranked)],
}
with open(RESULTS_BASE / 'final_summary.json', 'w') as f:
    json.dump(final_summary, f, indent=2)
print('✅ Resumen final guardado: final_summary.json')